In [1]:
## imports

# libraries
import os
import re
import glob
import numpy as np

## GetCentroid

In [2]:
# 标签数据椎骨中心获取

#获取labels文件内所有已分割椎体的中心RAS坐标，并转换为IJK坐标
def GetCentroid(file_mhd,file_RAS,file_IJK,is_clear):
    '''
    file_mhd: the label file for extracting centroid
    file_RAS: output path of the centroids of RAS
    file_IJK: output path of the centroids of IJK
    is_clear: if True, clear the hierarchy tree of 3D slicer
    '''
    
    loadedVolumeNode = slicer.util.loadVolume(file_mhd,{'name':'case_label', 'singleFile':True, 'labelmap':True, 'show':True})
    
    labelmapVolumeNode = getNode('case*')
    seg = slicer.mrmlScene.AddNewNodeByClass('vtkMRMLSegmentationNode','Segmentation')
    slicer.modules.segmentations.logic().ImportLabelmapToSegmentationNode(labelmapVolumeNode, seg)
    seg.CreateClosedSurfaceRepresentation()
    
    s = seg.GetSegmentation()
    Num = s.GetNumberOfSegments()
    markupsNode = slicer.mrmlScene.AddNewNodeByClass("vtkMRMLMarkupsFiducialNode",'MarkupsFiducial')
    com = vtk.vtkCenterOfMass()
    markupsNode.CreateDefaultDisplayNodes()
    centroid_list = []
    for i in range(0,Num,1):
        ss = s.GetNthSegment(i)
        pd = ss.GetRepresentation('Closed surface')
        com.SetInputData(pd)
        com.Update()
        centroid = list(com.GetCenter())
        markupsNode.AddFiducialFromArray(centroid,ss.GetName())
        centroid_list.append(centroid)
        
    #coordinates from RAS to IJK
    centroids = np.array(centroid_list)
    Mat = vtk.vtkMatrix4x4()
    labelmapVolumeNode.GetRASToIJKMatrix(Mat)
    print(Mat)
    centroidIJKlist = []
    for centroidRAS in centroids:
        centroidWorld = np.r_[centroidRAS,[1.0]]
        centroidIJK = list(Mat.MultiplyPoint(centroidWorld))
        centroidIJK.pop()
        centroidIJKlist.append(centroidIJK)

    np_centroidIJK = np.array(centroidIJKlist)
    
    path_RAS = os.path.dirname(file_RAS)
    if not os.path.exists(path_RAS):
        os.makedirs(path_RAS)
    path_IJK = os.path.dirname(file_IJK)    
    if not os.path.exists(path_IJK):
            os.makedirs(path_IJK)
    
    np.savetxt(file_RAS, centroids, delimiter=',')
    np.savetxt(file_IJK, np_centroidIJK, delimiter=',')

    if is_clear:
        slicer.mrmlScene.RemoveNode(labelmapVolumeNode)
        slicer.mrmlScene.RemoveNode(seg)
        slicer.mrmlScene.RemoveNode(markupsNode)

    return np_centroidIJK

In [3]:
#Windows
#rootPath = "E:/Dataset from SpineWeb/Dataset2-Jack"
#Linux
#rootPath = "/home/shu/Dataset/Dataset-xVertSeg"
rootPath = "/home/shu/Dataset/TestHealthyCases"

## 根据labels获取椎骨中心

In [6]:
#获得labels所在的文件所有路径
SrcmhdFolder = '/Resample1' #'/Resample' #'/Source'
labels = sorted(glob.glob(rootPath + SrcmhdFolder + '/masks/*.mhd'))[2:3]

In [7]:
labels

['/home/shu/Dataset/TestHealthyCases/Resample1/masks/Resmpl_case20_label.mhd']

In [8]:
#检测所有训练数据集label的椎骨中心并保存到相应文件

for mhd in labels:
    filename = re.sub(r'.mhd', '',os.path.split(mhd)[1])
    file_RAS = rootPath + SrcmhdFolder + '/Centroids/RAS/'+filename+'.csv'
    file_IJK = rootPath + SrcmhdFolder + '/Centroids/IJK/'+filename+'.csv'
    
    centroids_IJK = GetCentroid(mhd,file_RAS,file_IJK,True)

vtkMatrix4x4 (0x8ff14e0)
  Debug: Off
  Modified Time: 701323
  Reference Count: 1
  Registered Events: (none)
  Elements:
    -1 -0 0 -0 
    -0 -1 -0 0 
    0 -0 1 -0 
    -0 0 -0 1 


